In [1]:
import boto3
session = boto3.Session()

bedrock = session.client("bedrock", region_name="us-east-1")
br = session.client("bedrock-runtime", region_name="us-east-1")

In [5]:
import dspy

region_name = "us-east-1"

lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    max_tokens=4096
)

In [6]:
dspy.settings.configure(lm=lm)

In [8]:
math = dspy.ChainOfThought("question -> answer: float")
answer = math(question="Two dice are tossed. What is the probability that the sum equals two?")

In [11]:
print(answer.reasoning)

Let's solve this step by step:
1. For the sum to equal 2, we need both dice to show 1
2. Each die has 6 possible outcomes
3. Total possible outcomes when rolling 2 dice = 6 × 6 = 36
4. Favorable outcome is only one combination: (1,1)
5. Therefore probability = 1/36


In [16]:
for msg in math.history[0]['messages']:
    print(msg['content'])

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.
[[ ## question ## ]]
Two dice are tossed. What is the probability that the sum equals two?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]` (must be formatted as a valid Python float), and then ending with the marker for `[[ ## completed ## ]]`.


In [ ]:

# Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
qa = dspy.ChainOfThought('question -> answer')

# # Run with the default LM configured with `dspy.configure` above.
# response = qa(question="How many floors are in the castle David Gregory inherited?")
# print(response.answer)

# Setting the default LM
dspy.configure(lm=dspy.LM('bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0'))
response = qa(question="How many floors are in the castle David Gregory inherited?")
print('Sonnet V2', response.answer)

# Change the default LM for a single module.
with dspy.context(lm=dspy.LM('bedrock/us.anthropic.claude-3-haiku-20240307-v1:0')):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('Haiku V1:', response.answer)

Unknown - there is not enough information provided to determine the number of floors in David Gregory's inherited castle.
Sonnet V2 Unknown - there is not enough information provided to determine the number of floors in David Gregory's inherited castle.
Haiku V1: I'm sorry, but I don't have enough information to answer how many floors are in the castle that David Gregory inherited. The question does not provide any details about this castle.


In [19]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")


task = "context:str, text: str -> faithfulness: bool, evidence: dict[str, list[str]]"

solver = dspy.ChainOfThought(task)

In [ ]:
from pydantic import BaseModel

class ContextQuery(BaseModel):
    context: str
    query: str


class QueryResponse(BaseModel):
    response: str
    evidence: str

solver = dspy.Predict('query: ContextQuery -> response: QueryResponse')

context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."

text = "Lee scored 3 goals for Colchester United."

query_1 = ContextQuery(
    context=context,
    query=text
)

response = solver(query = query_1)



In [ ]:
class TestObj:
    def __init__(self) -> None:
        pass

    def forward(self, x: int) -> int:
        return x + 1

    def __call__(self, x: int) -> int:
        return self.forward(x)

test_obj = TestObj()

test_obj(1)

2

In [43]:
print(response.response)

response='False' evidence="He scored twice for the U's but was unable to save them from relegation."


In [44]:
for msg in solver.history[0]['messages']:
    print(msg['content'])

Your input fields are:
1. `query` (ContextQuery):
Your output fields are:
1. `response` (QueryResponse):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## query ## ]]
{query}

[[ ## response ## ]]
{response}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "properties": {"evidence": {"type": "string", "title": "Evidence"}, "response": {"type": "string", "title": "Response"}}, "required": ["response", "evidence"], "title": "QueryResponse"}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `query`, produce the fields `response`.
[[ ## query ## ]]
{"context": "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for 

## Optimizing Out of Scope Prompt

```xml
<ROLE>
    You are an expert email classification agent specializing in identifying Out of Scope communications from email communications. Your task is to analyze email content and determine if it constitutes external vendor communications, recruitment agency solicitations, spam/phishing attempts, or other communications that should not be processed by internal systems.
</ROLE>
<OBJECTIVE>
    Classify whether the given email falls under the category of "Out of Scope" based on the presence of external vendors, recruitment agencies, spam/phishing attempts, marketing communications, or other non-internal business communications that should be filtered out.
</OBJECTIVE>
<MODEL_INSTRUCTIONS>
    - Out of scope communications should be filtered out from internal processing 
    - Spam/phishing attempts require immediate blocking and security review 
    - Vendor communications should be routed to appropriate procurement teams 
    - Recruitment communications should be handled by HR/recruitment teams 
    - Internal employee communications should always be processed 
    - External domains are strong indicators of out of scope communications 
    - Marketing language and promotional offers indicate external communications 
    - Confidence scores help determine filtering accuracy 
    - Escalation flags help route security threats appropriately 
</MODEL_INSTRUCTIONS>
<RESPONSE_INSTRUCTIONS>
Respond ONLY with the XML classification result
</RESPONSE_INSTRUCTIONS>

In [ ]:
from typing import Annotated, Optional
from pydantic import Field
import dspy

region_name = "us-east-1"

lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    max_tokens=4096
)


class EmailContent(BaseModel):
    body: Annotated[str, Field(..., description="The body of the email")]
    subject: Annotated[str, Field(..., description="The subject of the email")]


class ClassificationOutput(BaseModel):
    is_positive: Annotated[bool, Field(..., description="Whether the email is classified belongs to the positive class")]
    confidence_score: Annotated[float, Field(..., description="The confidence score for the classification")]
    rationale: Annotated[str, Field(..., description="The rationale for the classification")]


class BaseClassificationModule(dspy.Module):
    def __init__(self, classification_type: str):
        self.classification_type = classification_type
        self.role = dspy.

    def forward(self, email: EmailContent) -> ClassificationOutput:
        pass

In [ ]:
import dspy

class OptimizePromptSignature(dspy.Signature):
    """
    Optimizes a given prompt by structuring it into detailed, actionable components.
    """
    unoptimized_prompt = dspy.InputField(
        desc="The initial, unstructured, or semi-structured prompt to be optimized."
    )

    role = dspy.OutputField(
        desc="Define the persona or role the AI should adopt. (e.g., 'You are a helpful assistant specializing in creative writing.')"
    )
    objective = dspy.OutputField(
        desc="Clearly state the primary goal or task the AI needs to accomplish."
    )
    model_instructions = dspy.OutputField(
        desc="Provide detailed, step-by-step instructions for the AI to follow to meet the objective."
    )
    positive_scenarios = dspy.OutputField(
        desc="Describe ideal outcomes or what success looks like. Include examples of high-quality responses."
    )
    examples = dspy.OutputField(
        desc="Provide concrete input/output examples to guide the AI's behavior and response format."
    )
    confidence_score_criteria = dspy.OutputField(
        desc="Define the criteria for a confidence score (1-5), where 5 is highest, to help the model self-assess its response quality."
    )
    task_execution_plan = dspy.OutputField(
        desc="Outline a clear, sequential plan for how the AI should approach and execute the task."
    )


# --- 2. Create the DSPy Module for Optimization ---
# This module uses the signature to create a predictor that will perform the optimization.

class PromptOptimizer(dspy.Module):
    """A module to optimize prompts into a structured format."""
    def __init__(self):
        super().__init__()
        # The core of this module is a dspy.Predict module, which is initialized with our signature.
        self.optimizer = dspy.Predict(OptimizePromptSignature)

    def forward(self, prompt_text):
        """
        Executes the prompt optimization.

        Args:
            prompt_text (str): The unoptimized prompt text.

        Returns:
            A dspy.Prediction object containing the structured, optimized prompt fields.
        """
        return self.optimizer(unoptimized_prompt=prompt_text)


 

In [ ]:
model_versions = [
    "us.anthropic.claude-3-haiku-20240307-v1:0",
    "us.anthropic.claude-3-opus-20240229-v1:0",
    "us.anthropic.claude-3-sonnet-20240229-v1:0",
    "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "us.anthropic.claude-3-5-sonnet-20240620-v1:0",
    "us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    "us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    "us.anthropic.claude-opus-4-20250514-v1:0",
    "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "us.deepseek.r1-v1:0",
    "us.meta.llama4-maverick-17b-instruct-v1:0",
    "us.meta.llama4-scout-17b-instruct-v1:0",
    "us.meta.llama3-1-70b-instruct-v1:0",
    "us.meta.llama3-1-8b-instruct-v1:0",
    "us.meta.llama3-2-11b-instruct-v1:0",
    "us.meta.llama3-2-1b-instruct-v1:0",
    "us.meta.llama3-2-3b-instruct-v1:0",
    "us.meta.llama3-2-90b-instruct-v1:0",
    "us.meta.llama3-3-70b-instruct-v1:0",
    "us.mistral.pixtral-large-2502-v1:0",
    "us.amazon.nova-lite-v1:0",
    "us.amazon.nova-micro-v1:0",
    "us.amazon.nova-premier-v1:0",
    "us.amazon.nova-pro-v1:0",
]

In [1]:
import litellm

In [ ]:
from litellm import completion


response = completion(
  model="bedrock/us.anthropic.claude-3-sonnet-20240229-v1:0",
  messages=[{ "content": "Hello, how are you?","role": "user"}],
  max
)

print(response)

ModelResponse(id='chatcmpl-ebbec173-64d0-4a84-a856-e75566628fdb', created=1754392077, model='us.anthropic.claude-3-sonnet-20240229-v1:0', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="Hello! As an AI language model, I don't have subjective experiences like feelings, but I'm operating properly and ready to assist you with any questions or tasks you might have. How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None))], usage=Usage(completion_tokens=47, prompt_tokens=13, total_tokens=60, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0))


In [9]:
print(response.choices[0].message.content)

Hello! As an AI language model, I don't have subjective experiences like feelings, but I'm operating properly and ready to assist you with any questions or tasks you might have. How can I help you today?
